# 🤖 LLM Agents com HuggingFace — smolagents
### Atividade #8 – Indo Além

---

## 🧠 O que é um LLM Agent?

Um **LLM Agent** (Agente de Modelo de Linguagem) é um sistema em que um **modelo de linguagem** (LLM) atua como o "cérebro" de um fluxo de decisões. Em vez de apenas gerar texto, o agente:

1. **Recebe** uma tarefa em linguagem natural
2. **Decide** quais ferramentas usar (calculadora, busca, API, etc.)
3. **Executa** as ferramentas e observa os resultados
4. **Itera** até chegar na resposta final

Neste notebook usaremos a biblioteca **`smolagents`** do HuggingFace para construir um agente simples capaz de:
- Realizar cálculos matemáticos
- Contar palavras em um texto
- Converter temperaturas (Celsius ↔ Fahrenheit)

---

## 📦 Passo 1: Instalação das Bibliotecas

In [ ]:
# Instalamos o smolagents (biblioteca oficial de agentes do HuggingFace)
# e o huggingface_hub para autenticação com a API
!pip install smolagents huggingface_hub -q

## 🔑 Passo 2: Autenticação no HuggingFace

Para usar os modelos via Inference API, precisamos de um token do HuggingFace.

👉 Crie sua conta gratuita em: https://huggingface.co  
👉 Gere seu token em: https://huggingface.co/settings/tokens

In [ ]:
import os

# Cole seu token do HuggingFace aqui
# Você pode criar um token gratuito em: https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = "hf_SEU_TOKEN_AQUI"  # ← substitua pelo seu token

# Confirmação
print("✅ Token configurado!")

## 🛠️ Passo 3: Importações

In [ ]:
# smolagents: biblioteca oficial de agentes do HuggingFace
from smolagents import CodeAgent, tool, InferenceClientModel

print("✅ Bibliotecas importadas com sucesso!")

## 🔧 Passo 4: Criando as Ferramentas (Tools)

**Ferramentas** são funções Python que o agente pode chamar para executar tarefas específicas.  
O decorador `@tool` informa ao agente que aquela função está disponível para uso.  

> ⚠️ O **docstring** (texto entre `"""`) é obrigatório: é por ele que o agente entende o que a ferramenta faz!

In [ ]:
@tool
def calculadora(expressao: str) -> str:
    """
    Avalia uma expressão matemática e retorna o resultado.
    Use para somar, subtrair, multiplicar, dividir ou calcular potências.
    Exemplo de entrada: '25 * 4 + 10'

    Args:
        expressao: a expressão matemática como string, por exemplo '2 + 2' ou '10 * 5'
    """
    try:
        resultado = eval(expressao)  # avalia a expressão matematicamente
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Erro ao calcular: {str(e)}"


@tool
def contar_palavras(texto: str) -> str:
    """
    Conta o número de palavras em um texto.
    Use quando precisar saber quantas palavras tem uma frase ou parágrafo.

    Args:
        texto: o texto do qual se deseja contar as palavras
    """
    palavras = texto.split()           # separa as palavras por espaço
    total = len(palavras)              # conta quantas existem
    return f"O texto possui {total} palavra(s)."


@tool
def celsius_para_fahrenheit(celsius: float) -> str:
    """
    Converte uma temperatura de Celsius para Fahrenheit.
    Use quando precisar converter temperaturas entre as duas escalas.

    Args:
        celsius: a temperatura em graus Celsius (número)
    """
    fahrenheit = (celsius * 9 / 5) + 32   # fórmula de conversão
    return f"{celsius}°C = {fahrenheit:.1f}°F"


print("✅ Ferramentas criadas:")
print("   🔢 calculadora")
print("   📝 contar_palavras")
print("   🌡️  celsius_para_fahrenheit")

## 🤖 Passo 5: Criando o Agente

Agora conectamos tudo: o **modelo de linguagem** + as **ferramentas** = nosso agente!

- `InferenceClientModel`: conecta com o modelo via API do HuggingFace (sem precisar rodar localmente)
- `CodeAgent`: tipo de agente que gera código Python para chamar as ferramentas

In [ ]:
# Escolhemos o modelo que vai "pensar" por nosso agente
# Qwen2.5-72B-Instruct é um LLM poderoso disponível gratuitamente no HuggingFace
modelo = InferenceClientModel(
    model_id="Qwen/Qwen2.5-72B-Instruct",
    token=os.environ["HF_TOKEN"]
)

# Criamos o agente, passando o modelo e as ferramentas disponíveis
agente = CodeAgent(
    tools=[calculadora, contar_palavras, celsius_para_fahrenheit],
    model=modelo,
    verbose=True   # mostra o raciocínio passo a passo do agente
)

print("✅ Agente criado e pronto para uso!")

## 🚀 Passo 6: Testando o Agente

Agora vamos ver o agente em ação! Ele irá:
1. Ler nossa pergunta
2. Decidir quais ferramentas usar
3. Executar as ferramentas
4. Combinar os resultados para responder

In [ ]:
# ───────────────────────────────────────────────
# Tarefa 1: Problema que exige calculadora
# ───────────────────────────────────────────────
print("=" * 60)
print("🧮 TAREFA 1: Cálculo matemático")
print("=" * 60)

resposta1 = agente.run(
    "Se eu tenho 15 laranjas e compro mais 8, depois dou metade para um amigo, com quantas fico?"
)

print("\n📌 Resposta final:", resposta1)

In [ ]:
# ───────────────────────────────────────────────
# Tarefa 2: Problema que exige contagem de palavras
# ───────────────────────────────────────────────
print("=" * 60)
print("📝 TAREFA 2: Contagem de palavras")
print("=" * 60)

frase = "A inteligência artificial está transformando o mundo de maneiras incríveis e surpreendentes"
resposta2 = agente.run(
    f"Quantas palavras tem a seguinte frase? '{frase}' E qual é o quadrado desse número?"
)

print("\n📌 Resposta final:", resposta2)

In [ ]:
# ───────────────────────────────────────────────
# Tarefa 3: Problema que combina múltiplas ferramentas
# ───────────────────────────────────────────────
print("=" * 60)
print("🌡️  TAREFA 3: Conversão de temperatura + cálculo")
print("=" * 60)

resposta3 = agente.run(
    "A temperatura em São Paulo hoje é 28°C. Converta para Fahrenheit. "
    "Depois calcule a diferença entre esse valor em Fahrenheit e 100."
)

print("\n📌 Resposta final:", resposta3)

## 📊 Passo 7: Entendendo o que aconteceu

Veja como o agente se comportou em cada tarefa:

| Tarefa | Ferramentas usadas | Comportamento do agente |
|--------|-------------------|-------------------------|
| Laranjas | `calculadora` | Identificou que precisava fazer (15+8)/2 |
| Contagem + quadrado | `contar_palavras` + `calculadora` | Encadeou 2 ferramentas em sequência |
| Temperatura | `celsius_para_fahrenheit` + `calculadora` | Usou 2 ferramentas com dependência entre elas |

Isso demonstra o poder dos **LLM Agents**: o modelo **raciocina** e **orquestra** as ferramentas automaticamente, sem precisarmos dizer explicitamente qual ferramenta usar!

## 🏁 Conclusão

Neste notebook aprendemos a:

✅ Entender o conceito de **LLM Agents**  
✅ Criar **ferramentas personalizadas** com o decorador `@tool`  
✅ Usar o modelo **Qwen2.5-72B** via HuggingFace Inference API  
✅ Construir um **CodeAgent** capaz de raciocinar e encadear ferramentas  
✅ Ver o agente resolver problemas reais de forma autônoma  

---

## 📚 Referências

- [smolagents – Documentação Oficial](https://huggingface.co/docs/smolagents)
- [HuggingFace – Plataforma de Modelos](https://huggingface.co)
- [Qwen2.5-72B-Instruct no HuggingFace](https://huggingface.co/Qwen/Qwen2.5-72B-Instruct)
- [Guia: Building Agents with smolagents](https://huggingface.co/blog/smolagents)
- [HuggingFace Inference API](https://huggingface.co/docs/api-inference)